# 04. Feature Engineering & ML Matching Model

**Team Role**: Member 3

### Purpose:
This notebook covers the extraction of pairwise similarity features (name, address, token, character, country) from candidate pairs, model training, cross-validation, and decision threshold calibration aimed at maximizing the competition F0.5 score.

In [1]:
# ============================================
# MEMBER 3 - ITERATION 1
# Feature Engineering & ML Matching
# ============================================

import os
import sys
import pandas as pd
import numpy as np

print("Python environment ready!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python environment ready!
Pandas version: 2.2.3
NumPy version: 2.1.3


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

zip_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/6ab10eb3b23ba_student_resource.zip"

print("ZIP exists:", os.path.exists(zip_path))
print("ZIP size (GB):", os.path.getsize(zip_path) / (1024**3) if os.path.exists(zip_path) else "Not found")

ZIP exists: True
ZIP size (GB): 1.0196335818618536


In [5]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/6ab10eb3b23ba_student_resource.zip"

extract_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")
print("Extracted to:", extract_path)

Extraction complete!
Extracted to: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted


In [6]:
import os

for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)

    if level > 3:
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}    {file}")

extracted/
    student_resource/
        Documentation_template.md
        .DS_Store
        README.md
        dataset/
            .DS_Store
            test/
                test_source2.tsv
                test_source3.tsv
                test_source1.tsv
            train/
                train_ground_truth.tsv
                train_source2.tsv
                train_source3.tsv
                train_source1.tsv
        utils/
            validate_submission.py
    __MACOSX/
        ._student_resource
        student_resource/
            ._Documentation_template.md
            ._.DS_Store
            ._dataset
            ._utils
            ._README.md
            dataset/
                ._.DS_Store
                ._test
                ._train
            utils/
                ._validate_submission.py


In [7]:
# ============================================
# Challenge dataset paths
# ============================================

BASE_PATH = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource"

TRAIN_PATH = f"{BASE_PATH}/dataset/train"
TEST_PATH = f"{BASE_PATH}/dataset/test"

print("Training path:", TRAIN_PATH)
print("Test path:", TEST_PATH)

Training path: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource/dataset/train
Test path: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource/dataset/test


In [9]:
import pandas as pd

train_source1 = pd.read_csv(
    f"{TRAIN_PATH}/train_source1.tsv",
    sep="\t",
    dtype=str
)

train_source2 = pd.read_csv(
    f"{TRAIN_PATH}/train_source2.tsv",
    sep="\t",
    dtype=str
)

train_source3 = pd.read_csv(
    f"{TRAIN_PATH}/train_source3.tsv",
    sep="\t",
    dtype=str
)

train_ground_truth = pd.read_csv(
    f"{TRAIN_PATH}/train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("Source 1:", train_source1.shape)
print("Source 2:", train_source2.shape)
print("Source 3:", train_source3.shape)
print("Ground truth:", train_ground_truth.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground truth: (2206821, 2)


In [10]:
print("SOURCE 1 COLUMNS")
print(train_source1.columns.tolist())

print("\nSOURCE 2 COLUMNS")
print(train_source2.columns.tolist())

print("\nSOURCE 3 COLUMNS")
print(train_source3.columns.tolist())

print("\nGROUND TRUTH COLUMNS")
print(train_ground_truth.columns.tolist())

SOURCE 1 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

SOURCE 2 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

SOURCE 3 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

GROUND TRUTH COLUMNS
['source1_entity_id', 'matched_entity_ids']


In [11]:
display(train_source1.head(5))

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [12]:
display(train_source2.head(5))

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [13]:
display(train_source3.head(5))

,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [14]:
display(train_ground_truth.head(10))

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [15]:
# ============================================
# Member 3 - Development sample
# ============================================

DEV_N = 10_000

dev_source1 = train_source1.head(DEV_N).copy()

print("Development Source 1:", dev_source1.shape)
print("Full Source 1:", train_source1.shape)

Development Source 1: (10000, 4)
Full Source 1: (2206821, 4)


In [16]:
dev_source2 = train_source2.copy()
dev_source3 = train_source3.copy()

print("Source 2:", dev_source2.shape)
print("Source 3:", dev_source3.shape)

Source 2: (5034616, 4)
Source 3: (5285603, 4)


In [17]:
%cd /content

!git clone -b member3-model https://github.com/niharikagadhiraju12-boop/DataFlux.git

/content
Cloning into 'DataFlux'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 47 (delta 7), reused 43 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 8.90 MiB | 17.16 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [18]:
import sys

PROJECT_SRC = "/content/DataFlux/code/business_entity_resolution/src"

if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

print("Project source path added:")
print(PROJECT_SRC)

Project source path added:
/content/DataFlux/code/business_entity_resolution/src


In [19]:
from blocking import generate_candidates

print("Member 2 blocking module imported successfully!")

Member 2 blocking module imported successfully!


In [20]:
import inspect

print(inspect.signature(generate_candidates))

(source1_df: pandas.core.frame.DataFrame, target_df: pandas.core.frame.DataFrame, use_rare_tokens: bool = True, use_tfidf: bool = True, use_address_tokens: bool = True, k_top: int = 25, min_similarity: float = 0.18, max_token_freq: int = 150, max_addr_freq: int = 30) -> pandas.core.frame.DataFrame


In [21]:
import inspect

source_code = inspect.getsource(generate_candidates)
print(source_code)

def generate_candidates(
    source1_df: pd.DataFrame,
    target_df: pd.DataFrame,
    use_rare_tokens: bool = True,
    use_tfidf: bool = True,
    use_address_tokens: bool = True,
    k_top: int = 25,
    min_similarity: float = 0.18,
    max_token_freq: int = 150,
    max_addr_freq: int = 30,
) -> pd.DataFrame:
    """
    Execute the multi-pass blocking pipeline:
        1. Country partitioning
        2. Rare name-token blocking
        3. Character n-gram TF-IDF retrieval
        4. Distinctive address-token blocking (optional)
        5. Union and deduplication
    
    Guarantees:
        - Candidates are strictly S1-S2 or S1-S3 (no S1-S1 self-matches).
        - Multiple candidates per S1 are supported.
        - Output is deterministically sorted for reproducibility.

    Parameters:
        source1_df: DataFrame containing Source 1 records.
        target_df: DataFrame containing Target (Source 2 and/or Source 3) records.
        use_rare_tokens: Whether to enable rare toke

In [22]:
TEST_S1 = dev_source1.head(1000).copy()
TEST_S2 = dev_source2.head(100_000).copy()

print("Test Source 1:", TEST_S1.shape)
print("Test Source 2:", TEST_S2.shape)

Test Source 1: (1000, 4)
Test Source 2: (100000, 4)


In [23]:
candidates_test = generate_candidates(
    source1_df=TEST_S1,
    target_df=TEST_S2
)

print("Candidate pairs generated:", len(candidates_test))
print(candidates_test.head())

Candidate pairs generated: 69039
  source1_entity_id candidate_entity_id
0      S1-100146655        S2-138958741
1      S1-100146655        S2-165008290
2      S1-100146655        S2-226521083
3      S1-100146655        S2-229962546
4      S1-100146655        S2-234426829


In [24]:
# Prepare ground truth for the 1,000 Source 1 development records

GT_TEST = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(TEST_S1["entity_id"])
].copy()

print("Ground-truth rows:", len(GT_TEST))
print(GT_TEST.head())

Ground-truth rows: 1000
     source1_entity_id                                 matched_entity_ids
1371      S1-844896591                           S2-362218588,S3-11966366
2917      S1-548116192                                                NaN
5400      S1-378978603             S2-319300693,S3-948532405,S3-921016187
7826      S1-447452795                          S2-995233298,S3-366674728
9668      S1-401761505  S2-422370961,S2-415623223,S3-17464132,S3-16723...


In [25]:
# Build the set of true Source 1 -> Source 2 pairs

true_s2_pairs = set()

for _, row in GT_TEST.iterrows():
    matched_ids = row["matched_entity_ids"]

    if pd.isna(matched_ids):
        continue

    for entity_id in str(matched_ids).split(","):
        entity_id = entity_id.strip()

        if entity_id.startswith("S2-"):
            true_s2_pairs.add(
                (row["source1_entity_id"], entity_id)
            )

# Convert generated candidates into a set for fast lookup
candidate_pairs_set = set(
    zip(
        candidates_test["source1_entity_id"],
        candidates_test["candidate_entity_id"]
    )
)

# Count how many true S2 pairs were captured
captured_s2_pairs = true_s2_pairs.intersection(candidate_pairs_set)

print("Total true S2 pairs:", len(true_s2_pairs))
print("Captured true S2 pairs:", len(captured_s2_pairs))

if len(true_s2_pairs) > 0:
    recall = len(captured_s2_pairs) / len(true_s2_pairs)
    print("S2 candidate recall:", round(recall, 4))

Total true S2 pairs: 1706
Captured true S2 pairs: 32
S2 candidate recall: 0.0188


In [26]:
# Check how many true S2 matches are actually present
# inside our 100,000-row test target dataset.

test_s2_ids = set(TEST_S2["entity_id"])

true_s2_in_test_target = {
    pair for pair in true_s2_pairs
    if pair[1] in test_s2_ids
}

print("True S2 pairs:", len(true_s2_pairs))
print("True S2 pairs present in TEST_S2:", len(true_s2_in_test_target))

if len(true_s2_pairs) > 0:
    print(
        "Fraction of true S2 pairs available to blocking:",
        round(len(true_s2_in_test_target) / len(true_s2_pairs), 4)
    )

True S2 pairs: 1706
True S2 pairs present in TEST_S2: 32
Fraction of true S2 pairs available to blocking: 0.0188


In [27]:
import psutil

memory = psutil.virtual_memory()

print("Total RAM (GB):", round(memory.total / (1024**3), 2))
print("Available RAM (GB):", round(memory.available / (1024**3), 2))
print("Used RAM (GB):", round(memory.used / (1024**3), 2))

Total RAM (GB): 12.67
Available RAM (GB): 7.01
Used RAM (GB): 5.36


In [28]:
TEST_S2_LARGE = dev_source2.head(1_000_000).copy()

print("Large test Source 2:", TEST_S2_LARGE.shape)

Large test Source 2: (1000000, 4)


In [29]:
test_s2_large_ids = set(TEST_S2_LARGE["entity_id"])

true_s2_in_large_target = {
    pair for pair in true_s2_pairs
    if pair[1] in test_s2_large_ids
}

print("Total true S2 pairs:", len(true_s2_pairs))
print("True S2 pairs present in 1M target:", len(true_s2_in_large_target))

print(
    "Fraction of true S2 pairs available:",
    round(len(true_s2_in_large_target) / len(true_s2_pairs), 4)
)

Total true S2 pairs: 1706
True S2 pairs present in 1M target: 348
Fraction of true S2 pairs available: 0.204


In [30]:
from blocking import tfidf_name_block
import inspect

print(inspect.signature(tfidf_name_block))
print()
print(inspect.getsource(tfidf_name_block))

(source1_df: pandas.core.frame.DataFrame, target_df: pandas.core.frame.DataFrame, k_top: int = 25, min_similarity: float = 0.18, chunk_size: int = 500, ngram_range: Tuple[int, int] = (3, 4)) -> Set[Tuple[str, str]]

def tfidf_name_block(
    source1_df: pd.DataFrame,
    target_df: pd.DataFrame,
    k_top: int = 25,
    min_similarity: float = 0.18,
    chunk_size: int = 500,
    ngram_range: Tuple[int, int] = (3, 4),
) -> Set[Tuple[str, str]]:
    """
    Generate candidate pairs via character n-gram TF-IDF cosine similarity.

    Why: Character n-grams are robust against misspellings ('Wilblims' vs 'Williams'),
    diacritics/accents ('Nónet' vs 'Nonet'), concatenations ('maurewilliamscolombier.com'),
    and word transpositions.

    Uses batched matrix multiplication to ensure O(chunk_size) memory usage.

    Parameters:
        source1_df: Preprocessed Source 1 records.
        target_df: Preprocessed Target records.
        k_top: Number of top candidate matches to retrieve per S

In [31]:
TEST_S1_SMALL = TEST_S1.head(100).copy()

print("Small Source 1:", TEST_S1_SMALL.shape)
print("Large Source 2:", TEST_S2_LARGE.shape)

Small Source 1: (100, 4)
Large Source 2: (1000000, 4)


In [33]:
from blocking import normalize_for_blocking

TEST_S1_SMALL_NORM = normalize_for_blocking(TEST_S1_SMALL)
TEST_S2_LARGE_NORM = normalize_for_blocking(TEST_S2_LARGE)

print("Normalized Source 1 columns:")
print(TEST_S1_SMALL_NORM.columns.tolist())

print("\nNormalized Source 2 columns:")
print(TEST_S2_LARGE_NORM.columns.tolist())

Normalized Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'norm_country']

Normalized Source 2 columns:
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'norm_country']


In [34]:
tfidf_test = tfidf_name_block(
    source1_df=TEST_S1_SMALL_NORM,
    target_df=TEST_S2_LARGE_NORM,
    k_top=25,
    min_similarity=0.18,
    chunk_size=100
)

print("TF-IDF candidate pairs:", len(tfidf_test))

TF-IDF candidate pairs: 2500


In [36]:
import features

print("Features module imported successfully!")
print(features.__file__)

Features module imported successfully!
/content/DataFlux/code/business_entity_resolution/src/features.py


In [37]:
import inspect

print(inspect.getsource(features))

"""
Feature Engineering Module for Business Entity Resolution.

Responsibilities:
- Extracting pairwise comparison features for candidate entity pairs
- Vectorizing entity attributes for ML classification / matching

POTENTIAL FEATURE CATEGORIES (Documented for planning purposes; not implemented yet):
-----------------------------------------------------------------------------------
1. Business-Name Similarity:
   - String distance metrics (Levenshtein distance, Jaro-Winkler, Damerau-Levenshtein)
   - Token-based metrics (Jaccard similarity, Cosine similarity on token counts/TF-IDF)
   - Length difference and prefix/suffix match ratios

2. Address Similarity:
   - Token overlap and Jaccard similarity across street/locality components
   - Character n-gram similarity (e.g., 3-gram / 4-gram overlap)
   - Normalized numerical tokens match (e.g., postal codes, street numbers)

3. Country Agreement:
   - Binary indicator of exact country match
   - Handling of missing/null country values



In [38]:
import importlib.util

print("rapidfuzz installed:",
      importlib.util.find_spec("rapidfuzz") is not None)

print("sklearn installed:",
      importlib.util.find_spec("sklearn") is not None)

rapidfuzz installed: False
sklearn installed: True


In [39]:
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher


def text_normalize(value):
    """Convert a value to a safe normalized string."""
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def token_jaccard(a, b):
    """Jaccard similarity between whitespace-separated token sets."""
    a_tokens = set(text_normalize(a).split())
    b_tokens = set(text_normalize(b).split())

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def edit_similarity(a, b):
    """Normalized character-level similarity."""
    a = text_normalize(a)
    b = text_normalize(b)

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


def length_difference(a, b):
    """Absolute difference in character lengths."""
    a = text_normalize(a)
    b = text_normalize(b)

    return abs(len(a) - len(b))


def extract_pair_features(
    candidate_pairs,
    source_a,
    source_b,
    **kwargs
):
    """
    Create numerical similarity features for candidate entity pairs.
    """

    # Keep only the columns needed for joining.
    a = source_a[
        ["entity_id", "business_name", "business_address", "country"]
    ].copy()

    b = source_b[
        ["entity_id", "business_name", "business_address", "country"]
    ].copy()

    # Rename columns so Source A and Source B are clearly distinguished.
    a = a.rename(columns={
        "entity_id": "source1_entity_id",
        "business_name": "name_a",
        "business_address": "address_a",
        "country": "country_a",
    })

    b = b.rename(columns={
        "entity_id": "candidate_entity_id",
        "business_name": "name_b",
        "business_address": "address_b",
        "country": "country_b",
    })

    # Attach the actual entity attributes to every candidate pair.
    df = candidate_pairs.merge(
        a,
        on="source1_entity_id",
        how="left"
    ).merge(
        b,
        on="candidate_entity_id",
        how="left"
    )

    # Name features
    df["name_exact"] = (
        df["name_a"].fillna("").astype(str).str.strip().str.lower()
        ==
        df["name_b"].fillna("").astype(str).str.strip().str.lower()
    ).astype(int)

    df["name_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    df["name_edit_similarity"] = [
        edit_similarity(a, b)
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    df["name_length_diff"] = [
        length_difference(a, b)
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    # Address features
    df["address_exact"] = (
        df["address_a"].fillna("").astype(str).str.strip().str.lower()
        ==
        df["address_b"].fillna("").astype(str).str.strip().str.lower()
    ).astype(int)

    df["address_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    df["address_edit_similarity"] = [
        edit_similarity(a, b)
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    df["address_length_diff"] = [
        length_difference(a, b)
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    # Country feature
    country_a = df["country_a"].fillna("").astype(str).str.strip().str.lower()
    country_b = df["country_b"].fillna("").astype(str).str.strip().str.lower()

    df["country_match"] = (
        (country_a != "") &
        (country_b != "") &
        (country_a == country_b)
    ).astype(int)

    # Token-count features
    df["name_token_count_diff"] = [
        abs(
            len(text_normalize(a).split())
            -
            len(text_normalize(b).split())
        )
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    df["address_token_count_diff"] = [
        abs(
            len(text_normalize(a).split())
            -
            len(text_normalize(b).split())
        )
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    return df

In [40]:
features_test = extract_pair_features(
    candidate_pairs=candidates_test,
    source_a=TEST_S1,
    source_b=TEST_S2
)

print("Feature dataframe shape:", features_test.shape)
print("\nFeature columns:")
print(features_test.columns.tolist())

print("\nFirst 5 rows:")
display(features_test.head())

Feature dataframe shape: (69039, 19)

Feature columns:
['source1_entity_id', 'candidate_entity_id', 'name_a', 'address_a', 'country_a', 'name_b', 'address_b', 'country_b', 'name_exact', 'name_jaccard', 'name_edit_similarity', 'name_length_diff', 'address_exact', 'address_jaccard', 'address_edit_similarity', 'address_length_diff', 'country_match', 'name_token_count_diff', 'address_token_count_diff']

First 5 rows:


,source1_entity_id,candidate_entity_id,name_a,address_a,country_a,name_b,address_b,country_b,name_exact,name_jaccard,name_edit_similarity,name_length_diff,address_exact,address_jaccard,address_edit_similarity,address_length_diff,country_match,name_token_count_diff,address_token_count_diff
0,S1-100146655,S2-138958741,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Bricklayers Union Local No,"200 CLARK LOOP, LOCKHAT, TX",US,0,0.600000,0.754717,1,0,0.0,0.385965,3,1,0,1
1,S1-100146655,S2-165008290,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,BRICKLAYERS 481 LOCAL,"125 115ND STREET, VILLAGE OF PLEASANT PRAIRIE, WI",US,0,0.400000,0.708333,6,0,0.0,0.329114,19,1,1,2
2,S1-100146655,S2-226521083,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Electricians Local Union 420 Inc,"149 23RD AVENUE, DICKINSON, ND",US,0,0.285714,0.677966,5,0,0.0,0.333333,0,1,1,1
3,S1-100146655,S2-229962546,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Painters Union Local No,"0216 W WOODLAND AVE, UNDERWOOD, MN",US,0,0.333333,0.520000,4,0,0.0,0.406250,4,1,0,0
4,S1-100146655,S2-234426829,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Bricklayers Local 197 LLC,"PHOENIX, AZ, 3650 ORANGE DR",US,0,0.333333,0.730769,2,0,0.0,0.140351,3,1,0,1


In [41]:
# Create binary labels for the candidate pairs

features_test["label"] = [
    int(
        (s1_id, candidate_id) in true_s2_pairs
    )
    for s1_id, candidate_id in zip(
        features_test["source1_entity_id"],
        features_test["candidate_entity_id"]
    )
]

print("Total candidate pairs:", len(features_test))
print("Positive matches:", features_test["label"].sum())
print("Negative pairs:", (features_test["label"] == 0).sum())

print("\nLabel distribution:")
print(features_test["label"].value_counts())

Total candidate pairs: 69039
Positive matches: 32
Negative pairs: 69007

Label distribution:
label
0    69007
1       32
Name: count, dtype: int64


In [42]:
import model

print("Model module:")
print(model.__file__)

print("\nExisting model.py:")
print(inspect.getsource(model))

Model module:
/content/DataFlux/code/business_entity_resolution/src/model.py

Existing model.py:
"""
Matching Model Module for Business Entity Resolution.

Responsibilities:
- Training and configuring the classification model for entity pair matching
- Generating match probability predictions for candidate pairs
- Deciding match status based on thresholding tuned for the target metric
- Serializing and loading trained models

NOTE: The model will eventually predict whether a candidate pair represents
the same business entity. Do not train or implement the final model yet.
"""

from pathlib import Path
from typing import Any, Dict, Optional, Union
import pandas as pd


class EntityMatchingModel:
    """
    Wrapper for the entity matching classification model.
    """

    def __init__(self, model_params: Optional[Dict[str, Any]] = None) -> None:
        """
        Initialize the matching model configuration.

        Parameters:
            model_params: Optional dictionary of hyperpa

In [43]:
from sklearn.linear_model import LogisticRegression


# Numerical features used by the first model
FEATURE_COLUMNS = [
    "name_exact",
    "name_jaccard",
    "name_edit_similarity",
    "name_length_diff",
    "address_exact",
    "address_jaccard",
    "address_edit_similarity",
    "address_length_diff",
    "country_match",
    "name_token_count_diff",
    "address_token_count_diff",
]

X_test = features_test[FEATURE_COLUMNS].copy()
y_test = features_test["label"].copy()

print("Feature matrix shape:", X_test.shape)
print("Label shape:", y_test.shape)
print("Positive labels:", int(y_test.sum()))
print("Negative labels:", int((y_test == 0).sum()))

Feature matrix shape: (69039, 11)
Label shape: (69039,)
Positive labels: 32
Negative labels: 69007


In [44]:
# First ML model for entity matching
# This is a smoke test using the current candidate sample.

matching_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

matching_model.fit(X_test, y_test)

print("Model training completed!")

Model training completed!


In [45]:
# Generate probability that each candidate pair is a true match

match_probabilities = matching_model.predict_proba(X_test)[:, 1]

features_test["match_probability"] = match_probabilities

print("Probability generation completed!")

print("\nProbability statistics:")
print(features_test["match_probability"].describe())

print("\nTop 10 candidate pairs by probability:")
display(
    features_test[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "match_probability",
            "label"
        ]
    ]
    .sort_values("match_probability", ascending=False)
    .head(10)
)

Probability generation completed!

Probability statistics:
count    6.903900e+04
mean     1.394782e-02
std      8.215436e-02
min      6.755243e-24
25%      3.513752e-07
50%      1.524612e-05
75%      3.927716e-04
max      1.000000e+00
Name: match_probability, dtype: float64

Top 10 candidate pairs by probability:


,source1_entity_id,candidate_entity_id,match_probability,label
27297,S1-452691275,S2-320133796,1.0,1
59092,S1-877733833,S2-974050437,1.0,1
57029,S1-842881175,S2-523911840,1.0,1
567,S1-104226194,S2-894147929,1.0,1
33363,S1-533701421,S2-434618903,1.0,1
11005,S1-220020462,S2-342952603,1.0,1
55759,S1-814475089,S2-238559336,1.0,1
20787,S1-376690332,S2-787570292,1.0,1
12037,S1-236312382,S2-69010626,1.0,1
51508,S1-761585587,S2-801229590,1.0,1


In [46]:
# Prepare the probability output for Member 4

model_output = features_test[
    [
        "source1_entity_id",
        "candidate_entity_id",
        "match_probability"
    ]
].copy()

print("Model output shape:", model_output.shape)
display(model_output.head())

# Save the smoke-test probabilities
MODEL_OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "Amazon_ML_Challenge_2026/"
    "student_resource/"
    "member3_model_probabilities_smoke_test.tsv"
)

model_output.to_csv(
    MODEL_OUTPUT_PATH,
    sep="\t",
    index=False
)

print("\nSaved to:")
print(MODEL_OUTPUT_PATH)

Model output shape: (69039, 3)


,source1_entity_id,candidate_entity_id,match_probability
0,S1-100146655,S2-138958741,0.448575
1,S1-100146655,S2-165008290,0.002390
2,S1-100146655,S2-226521083,0.001084
3,S1-100146655,S2-229962546,0.000786
4,S1-100146655,S2-234426829,0.000597



Saved to:
/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/member3_model_probabilities_smoke_test.tsv


In [47]:
import importlib
import features

importlib.reload(features)

print("Updated features.py loaded successfully!")
print(features.extract_pair_features)

Updated features.py loaded successfully!
<function extract_pair_features at 0x7972c6686520>


In [48]:
import importlib
import model

importlib.reload(model)

print("Updated model.py loaded successfully!")
print(model.EntityMatchingModel)

Updated model.py loaded successfully!
<class 'model.EntityMatchingModel'>


In [49]:
from model import EntityMatchingModel

# Create the project-level matching model
entity_model = EntityMatchingModel()

# Train on our smoke-test feature matrix
entity_model.fit(
    X_test,
    y_test
)

print("EntityMatchingModel training completed!")

EntityMatchingModel training completed!


In [50]:
project_probabilities = entity_model.predict_proba(X_test)[:, 1]

print("Probability generation completed!")
print("Number of probabilities:", len(project_probabilities))
print("Minimum probability:", project_probabilities.min())
print("Maximum probability:", project_probabilities.max())

Probability generation completed!
Number of probabilities: 69039
Minimum probability: 6.75524314044976e-24
Maximum probability: 1.0


In [51]:
# Test threshold-based predictions
predictions = entity_model.predict(
    X_test,
    threshold=0.5
)

print("Prediction count:", len(predictions))
print("Predicted matches:", int(predictions.sum()))
print("Predicted non-matches:", int((predictions == 0).sum()))

Prediction count: 69039
Predicted matches: 591
Predicted non-matches: 68448


In [52]:
MODEL_PATH = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/entity_matching_model.joblib"

# Save the trained model
entity_model.save(MODEL_PATH)

print("Model saved to:")
print(MODEL_PATH)

Model saved to:
/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/entity_matching_model.joblib


In [53]:
# Load the saved model
loaded_model = EntityMatchingModel.load(MODEL_PATH)

print("Model loaded successfully!")

# Generate probabilities from the loaded model
loaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]

# Verify that the loaded model gives the same probabilities
same_predictions = np.allclose(
    project_probabilities,
    loaded_probabilities
)

print("Probabilities match original model:", same_predictions)

Model loaded successfully!
Probabilities match original model: True
